In [1]:
import pandas as pd
import numpy as np

movies_df = pd.read_csv('data/movies.csv')
movies = movies_df.copy()

ratings = pd.read_csv('data/ratings.csv')
ratings = ratings.copy()

tags_df = pd.read_csv('data/tags.csv')
tags = tags_df.copy()

Filtering ratings

In [2]:
lowest_acceptable_mean_rating_score = 2.0 # Set to zero to disable filtering
lowest_acceptable_rating_count = 100

ratings_grouped = ratings.groupby('movieId')['rating'].mean().sort_values()
good_movie_scores = ratings_grouped[ratings_grouped >= lowest_acceptable_mean_rating_score].index
ratings_filtered = ratings[ratings["movieId"].isin(good_movie_scores)]

ratings_per_movie = ratings.groupby('movieId')['rating'].count()
popular_movie = ratings_per_movie[ratings_per_movie >= lowest_acceptable_rating_count].index
ratings_filtered = ratings_filtered[ratings_filtered["movieId"].isin(popular_movie)]

Splitting movie genres string with on-hot encoder 

In [3]:
genre_dummies = movies["genres"].str.get_dummies("|")
movies_genre_one_hot = pd.concat([movies, genre_dummies], axis=1)
movies_genre_one_hot = movies_genre_one_hot.drop(columns="genres")

Filtering movies without genres

In [4]:
movies_filtered = movies_genre_one_hot[movies_genre_one_hot["(no genres listed)"] == 0]
movies_filtered = movies_filtered.drop(columns="(no genres listed)")

Merging filtered movies and ratings dataframe

In [5]:
movies_ratings_merge = ratings_filtered.merge(movies_filtered, on="movieId", how="inner")
movies_ratings_merge

,userId,movieId,rating,timestamp,title,Action,Adventure,Animation,Children,Comedy,...,Film-Noir,Horror,IMAX,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
0,1,1,4.0,1225734739,Toy Story (1995),0,1,1,1,1,...,0,0,0,0,0,0,0,0,0,0
1,1,110,4.0,1225865086,Braveheart (1995),1,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0
2,1,158,4.0,1225733503,Casper (1995),0,1,0,1,0,...,0,0,0,0,0,0,0,0,0,0
3,1,260,4.5,1225735204,Star Wars: Episode IV - A New Hope (1977),1,1,0,0,0,...,0,0,0,0,0,0,1,0,0,0
4,1,356,5.0,1225735119,Forrest Gump (1994),0,0,0,0,1,...,0,0,0,0,0,1,0,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32911842,330975,8340,2.0,1091583256,Escape from Alcatraz (1979),0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
32911843,330975,8493,2.5,1091585709,Memphis Belle (1990),1,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0
32911844,330975,8622,4.0,1091581777,Fahrenheit 9/11 (2004),0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
32911845,330975,8665,3.0,1091581765,"Bourne Supremacy, The (2004)",1,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0


Rekommender Contet
"identify similarities between movies based on genres based filtering KNN och cosine similarity"
"one-hot encoding on genres and a KNN-Transform with cosine similarity"

In [12]:
from sklearn.metrics.pairwise import cosine_similarity

def recommend(movie_title, n=5):
    matches = movies_filtered[movies_filtered["title"] == movie_title]
    genre_matrix = movies_filtered.drop(columns=["movieId", "title"])
    if matches.empty:
        return "Movie not found"
    
    idx = matches.index[0]
    
    # 🔥 bara jämför EN film mot alla
    sim_scores = cosine_similarity(
        genre_matrix.iloc[idx].values.reshape(1, -1),
        genre_matrix
    )[0]
    
    # enumerera + sortera
    sim_scores = list(enumerate(sim_scores))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    
    sim_scores = sim_scores[1:n+1]
    movie_indices = [i[0] for i in sim_scores]
    
    return movies_filtered.iloc[movie_indices][["title"]]

recommend("Toy Story (1995)")

,title
2203,Antz (1998)
3021,Toy Story 2 (1999)
3654,"Adventures of Rocky and Bullwinkle, The (2000)"
3913,"Emperor's New Groove, The (2000)"
4781,"Monsters, Inc. (2001)"
